### Please remember to download the following subdataset from AMASS website: https://amass.is.tue.mpg.de/download.php. Note only download the <u>SMPL+H G</u> data.
* ACCD (ACCD)
* HDM05 (MPI_HDM05)
* TCDHands (TCD_handMocap)
* SFU (SFU)
* BMLmovi (BMLmovi)
* CMU (CMU)
* Mosh (MPI_mosh)
* EKUT (EKUT)
* KIT  (KIT)
* Eyes_Janpan_Dataset (Eyes_Janpan_Dataset)
* BMLhandball (BMLhandball)
* Transitions (Transitions_mocap)
* PosePrior (MPI_Limits)
* HumanEva (HumanEva)
* SSM (SSM_synced)
* DFaust (DFaust_67)
* TotalCapture (TotalCapture)
* BMLrub (BioMotionLab_NTroje)

### Unzip all datasets. In the bracket we give the name of the unzipped file folder. Please correct yours to the given names if they are not the same.

### Place all files under the directory **./amass_data/**. The directory structure shoud look like the following:  
./amass_data/  
./amass_data/ACCAD/  
./amass_data/BioMotionLab_NTroje/  
./amass_data/BMLhandball/  
./amass_data/BMLmovi/   
./amass_data/CMU/  
./amass_data/DFaust_67/  
./amass_data/EKUT/  
./amass_data/Eyes_Japan_Dataset/  
./amass_data/HumanEva/  
./amass_data/KIT/  
./amass_data/MPI_HDM05/  
./amass_data/MPI_Limits/  
./amass_data/MPI_mosh/  
./amass_data/SFU/  
./amass_data/SSM_synced/  
./amass_data/TCD_handMocap/  
./amass_data/TotalCapture/  
./amass_data/Transitions_mocap/  

**Please make sure the file path are correct, otherwise it can not succeed.**

In [1]:
import os
from amass_to_pose import *

paths = []
folders = []
dataset_names = []
for root, dirs, files in os.walk('../amass_data'):
#     print(root, dirs, files)
#     for folder in dirs:
#         folders.append(os.path.join(root, folder))
    folders.append(root)
    for name in files:
        dataset_name = root.split('/')[2]
        if dataset_name not in dataset_names:
            dataset_names.append(dataset_name)
        paths.append(os.path.join(root, name))
all_count = len(paths)

ModuleNotFoundError: No module named 'kingdon'

In [7]:
save_root = '../pose_data'
save_folders = [folder.replace('amass_data', 'pose_data') for folder in folders]
for folder in save_folders:
    os.makedirs(folder, exist_ok=True)
group_path = [[path for path in paths if name in path] for name in dataset_names]

In [8]:
import json
from tqdm import tqdm
from os.path import join as pjoin
import logging

logging.basicConfig(filename="process.log",)
log = logging.getLogger(__name__)

with open("index.json", "r") as f:
    index_list = json.load(f)     # list of dicts

total_amount = len(index_list)

path_map = {}
m_num = 0
for row in index_list:
    k = row['source_path']
    new_name   = row["new_name"]
    start_frame = row["start_frame"]
    end_frame   = row["end_frame"] + 2
    if "Eyes_Japan_Dataset" in k:
        start_frame += 3*EXFPS
        end_frame += 3*EXFPS
        m_num += 1
    if "MPI_HDM05" in k:
        start_frame += 3*EXFPS
        end_frame += 3*EXFPS
        m_num += 1
    if "TotalCapture" in k:
        start_frame += 1*EXFPS
        end_frame += 1*EXFPS
        m_num += 1
    if "MPI_Limits" in k:
        start_frame += 1*EXFPS
        end_frame += 1*EXFPS
        m_num += 1
    if "Transitions_mocap" in k:
        start_frame += int(0.5*EXFPS)
        end_frame += int(0.5*EXFPS)
        m_num += 1
    path_map[k] = (new_name, start_frame, end_frame)
print('Total modified sequences:', m_num)

    

Total modified sequences: 2552


In [10]:
import time
save_dir = './joints'
stat_map = None
cur_count = 0
for paths in group_path:
    dataset_name = paths[0].split('/')[2]
    pbar = tqdm(paths)
    pbar.set_description('Processing: %s'%dataset_name)
    fps = 0
    for path in pbar:
        save_path = path.replace('../amass_data', './pose_data')
        save_path = save_path[:-3] + 'npy'
        # print(save_path)
        try:
            new_name, start_frame, end_frame = path_map[save_path]
        except:
            log.info(f"Not in index.json, skipping: {path}")
            continue
        new_name = new_name.replace('npy', 'npz')
        rotors, fps = load_amass(path, start_frame, end_frame)
        if (not rotors) or (rotors[0].shape[1] < 3):
            print(f"{path} No valid frames in the last file.")
            continue
        # if (rotors[0].shape[1] < 15):
        #     print(f"{path} has too few frames: {rotors[0].shape[1]}")
        try:
            value_map, value_map_m, stat_map, key_map = process_rotors(rotors, stat_map)
        except Exception as e:
            print(rotors[0].shape)
            print(f"Error processing {path}: {e}")
            continue
        
        np.savez(pjoin(save_dir, new_name), **value_map)
        np.savez(pjoin(save_dir, 'M'+new_name), **value_map_m)
        
    cur_count += len(paths)
    print('Processed / All (fps %d): %d/%d'% (fps, cur_count, all_count) )
    time.sleep(0.5)

Processing: BMLhandball: 100%|██████████| 659/659 [00:01<00:00, 404.22it/s]


Processed / All (fps 120): 659/14055


Processing: ACCAD: 100%|██████████| 252/252 [00:05<00:00, 48.38it/s]


Processed / All (fps 120): 911/14055


Processing: TCD_handMocap: 100%|██████████| 62/62 [00:00<00:00, 141099.75it/s]


...

Processed / All (fps 120): 13305/14055


Processing: Eyes_Japan_Dataset:  69%|██████▊   | 514/750 [00:17<00:08, 26.88it/s]

with 1 sign flips


Processing: Eyes_Japan_Dataset: 100%|██████████| 750/750 [00:25<00:00, 29.54it/s]


Processed / All (fps 120): 14055/14055


In [11]:
def serialize_stats(stat_map, filename):
    """Convert the {name: (array, array, int)} map to JSON."""
    serializable = {}
    for key, (arr1, arr2, count) in stat_map.items():
        serializable[key] = {
            "mean": arr1.tolist(),
            "M2": arr2.tolist(),
            "t_count": int(count)
        }

    with open(filename, "w") as f:
        json.dump(serializable, f, indent=2)

serialize_stats(stat_map, "stats.json")